# 策略梯度（Policy Gradient）

上一章把策略优化当作黑盒问题：只知道参数 $\theta$ 对应的累计回报，却没有利用轨迹中每一步的状态、动作和奖励。

本章开始利用强化学习问题的结构，介绍 Policy Gradient 与 REINFORCE。


## 1. 强化学习的基本形式

在时刻 $t$，智能体观察状态 $s_t$，按照随机策略

$$a_t\sim\pi_\theta(\cdot\mid s_t)$$

选择动作。环境根据转移分布产生下一状态和即时奖励。

从时刻 $t$ 开始的折扣累计回报定义为

$$G_t=r_{t+1}+\gamma r_{t+2}+\gamma^2 r_{t+3}+\cdots,$$

其中 $\gamma\in[0,1]$ 为折扣因子。策略优化通常以最大化期望累计回报为目标：

$$J(\theta)=\mathbb{E}_{\tau\sim\pi_\theta}[G_0].$$


## 2. 状态价值与动作价值

状态价值函数定义为

$$V^\pi(s)=\mathbb{E}[G_t\mid s_t=s],$$

动作价值函数定义为

$$Q^\pi(s,a)=\mathbb{E}[G_t\mid s_t=s,a_t=a].$$

二者回答不同问题：$V^\pi(s)$ 描述“处于这个状态有多好”，$Q^\pi(s,a)$ 描述“在这个状态执行这个动作有多好”。


## 3. Policy Gradient Theorem

策略梯度定理给出一个重要结果：即使环境动力学未知，也可以通过策略自身的对数概率梯度构造 $J(\theta)$ 的梯度。典型形式为

$$\nabla_\theta J(\theta)=\mathbb{E}\left[Q^\pi(s_t,a_t)\nabla_\theta\log\pi_\theta(a_t\mid s_t)\right].$$

这意味着我们不需要对环境求导，只需要策略网络是可微的，并且能够从环境中采样轨迹。


## 4. REINFORCE：用 Monte Carlo 回报估计价值

REINFORCE 使用实际观测到的 $G_t$ 近似动作价值：

$$Q^\pi(s_t,a_t)\approx G_t.$$

于是梯度估计为

$$\hat g=\sum_t G_t\nabla_\theta\log\pi_\theta(a_t\mid s_t).$$

为了做梯度下降，可以定义 loss

$$L(\theta)=-\sum_t G_t\log\pi_\theta(a_t\mid s_t).$$


In [ ]:
def discounted_returns(rewards, gamma=0.99):
    """从一条 episode 的即时奖励计算每个时刻的折扣累计回报。"""
    G = 0.0
    returns = []
    for r in reversed(rewards):
        G = r + gamma * G
        returns.append(G)
    return list(reversed(returns))


## 5. 一个最小随机策略网络

对于离散动作，可以让神经网络输出 logits，再通过 categorical distribution 得到随机策略。


In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Categorical

class PolicyNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, action_dim)
        )

    def distribution(self, state):
        logits = self.net(state)
        return Categorical(logits=logits)

    def sample_action(self, state):
        dist = self.distribution(state)
        action = dist.sample()
        return action, dist.log_prob(action)


## 6. REINFORCE 的一次更新

一条 episode 结束后，已经知道每个时刻之后实际获得的奖励，因此可以计算所有 $G_t$，再进行一次策略更新。


In [ ]:
def reinforce_update(optimizer, log_probs, rewards, gamma=0.99):
    returns = torch.tensor(discounted_returns(rewards, gamma), dtype=torch.float32)
    log_probs = torch.stack(log_probs)
    loss = -(log_probs * returns).sum()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return float(loss.detach())


## 7. 为什么梯度方差很大

Monte Carlo 回报 $G_t$ 受到之后所有随机状态转移和随机动作影响，因此 REINFORCE 的梯度估计通常方差较大。

一个常见技巧是引入 baseline。只要 baseline 不依赖当前动作，就不会改变策略梯度期望。最常用的 baseline 是状态价值函数：

$$A^\pi(s,a)=Q^\pi(s,a)-V^\pi(s).$$

$A^\pi$ 称为 Advantage（优势函数），表示这个动作相对于当前状态下的平均水平究竟好多少。


## 8. REINFORCE with Baseline

如果用一个参数化函数 $v_\phi(s)$ 估计 $V^\pi(s)$，策略梯度可以写成

$$\hat g=\sum_t (G_t-v_\phi(s_t))\nabla_\theta\log\pi_\theta(a_t\mid s_t).$$

同时用 Monte Carlo target $G_t$ 训练价值网络：

$$L_V(\phi)=\sum_t(G_t-v_\phi(s_t))^2.$$

这已经具有 Actor-Critic 的基本形态：策略网络是 Actor，价值网络是 Critic。区别在于这里 Critic 仍使用完整 episode 的 Monte Carlo target。


## 9. On-policy 的含义

REINFORCE 是 on-policy 方法：用于更新策略的数据必须由当前或非常接近当前的策略产生。

策略参数发生明显变化后，旧轨迹不再严格服从新的 $\pi_\theta$，不能像普通监督学习数据那样长期反复使用。

这也是后面 off-policy 方法（如 TD3）引入 Replay Buffer 时需要改变学习目标和推导方式的原因。


## 10. Episodic Task 与 Truncation

需要区分真正的终止状态 `terminated=True` 与因为时间上限而截断 `truncated=True`。

真正终止状态之后的价值可以看作 0；但时间截断通常不意味着系统进入零价值吸收状态。如果把 truncation 错当 termination，会对价值估计产生偏差。


## 本章要点

- Policy Gradient 直接优化期望累计回报。
- 对数概率技巧使我们无需对环境动力学求导。
- REINFORCE 用 Monte Carlo 回报估计价值，简单但方差较大。
- 使用 $V(s)$ 作为 baseline 可以形成 Advantage，并降低梯度方差。
- REINFORCE 属于 on-policy、episode-level 更新方法。
- 下一章 Actor-Critic 将通过 TD bootstrapping 进一步提高价值估计与在线更新效率。
